<a href="https://colab.research.google.com/github/maryamsohail32/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maryamsohail32/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

##  **My lane and why:**

I'm choosing **Lane 2: Refresh / Content Opportunity Scoring**. My question is which pages FlyRank's reviewers should look at first when deciding what to refresh, expand, or protect. I picked this lane because I already ran the starter pipeline in Week 1, and saw a learned model beat a hand-written rule by roughly 3x on Precision@50 (0.240 vs 0.740). That gap tells me there's real, learnable signal in this data beyond simple heuristics — which makes this lane worth building on for the next 7 weeks.
This isn't just "train a model and see what happens" — it first requires defining what
counts as declining (versus seasonality or consolidation), what unit we're ranking (page
vs. site), how much review capacity actually exists, and what a wrong call costs. Skip
that framing, and a model can score well while still recommending pages nobody should
review.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

## **The question: decision, action, cost of a wrong call**

**Decision this improves:** which pages a content reviewer should prioritize out of a large inventory, given a fixed weekly review capacity (say, 50 pages a week).

**Unit of analysis:** a single page (content item) — each row we rank is one page, not a
whole client site or a single day.

**Who acts on it:** an SEO/content strategist or reviewer with limited time, choosing where to spend it first.

**The action taken:** refresh outdated content, expand thin content, or flag a strong page for protection/monitoring instead of touching it.

**Cost of a wrong call, made concrete:**
- A **false positive** (flagging a healthy page) wastes a reviewer's limited slot — if even 1 in 4 flagged pages turns out fine, that's 25% of weekly capacity spent for nothing.
- A **false negative** (missing a real decliner) means that page keeps losing visibility silently for another full review cycle — potentially weeks of avoidable traffic loss before anyone notices.

Because review capacity is fixed and scarce, this is why ranking quality at the *top* of the list (Precision@20, Precision@50) matters far more than overall accuracy across the whole page inventory.

## **Task type and frame**

**Task type:** Ranking/scoring — the question is "which pages first," so the right metric
is Precision@K (K = weekly review capacity), not overall accuracy.

**One-paragraph frame:** For an SEO/content reviewer, deciding which pages to review first
out of a large inventory, we will build a ranked priority list from observable content and
search signals, scoring pages by likelihood of being a genuine refresh/expansion candidate,
measured by Precision@50. A wrong call costs wasted reviewer time (false positive) or a
silently worsening page (false negative). A plain if/else rule isn't enough because the
signal is spread across many correlated fields (staleness, visibility, position, CTR,
engagement) that shift together in ways too tangled to hand-write - which the baseline vs.
model gap (0.240 vs 0.740) already demonstrates. We will claim only observed, directional,
decision-support results.

**Honest caveat on the label:** the starter dataset's is_declining_label (trend_direction
== "down") is a current-window bucket, not a future observed outcome - a beginner proxy
label, not the ideal target. A stronger version of this lane would define the label from a
later time window (e.g. features from the prior 90 days predicting decline over the next 30
days), which I'll aim for once I move to the full warehouse release.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [1]:
import os, sys, subprocess

REPO_URL = "https://github.com/maryamsohail32/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if "google.colab" in sys.modules:
    os.chdir("/content")
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — check repo cloned correctly"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship
Starter data found. You're ready.


In [2]:
import pandas as pd
import json

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Number 1: how many pages are currently declining
declining_rate = df["trend_direction"].str.lower().eq("down").mean()
print(f"1) Share of pages currently declining: {declining_rate:.1%}")

# Number 2: pages that are stale AND still visible (core Lane 2 signal)
stale_visible = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500))
print(f"2) Share of pages stale (180+ days) AND still getting 500+ impressions: {stale_visible.mean():.1%}")

# Number 3: stacking both conditions - stale, visible, AND declining at once
stale_visible_declining = (stale_visible & df["trend_direction"].str.lower().eq("down"))
print(f"3) Share of pages that are stale, visible, AND declining right now: {stale_visible_declining.mean():.1%}")
print("Raw count stale+visible:", stale_visible.sum())
print("Raw count stale+visible+declining:", stale_visible_declining.sum())
print(df["days_since_last_update"].describe())

1) Share of pages currently declining: 54.2%
2) Share of pages stale (180+ days) AND still getting 500+ impressions: 0.1%
3) Share of pages that are stale, visible, AND declining right now: 0.1%
Raw count stale+visible: 17
Raw count stale+visible+declining: 16
count    30000.000000
mean        46.098300
std         42.078709
min          1.000000
25%         20.000000
50%         20.000000
75%        104.000000
max        373.000000
Name: days_since_last_update, dtype: float64


**Note:** Only 17 of 30,000 pages clear the stale (180+ days) AND visible (500+ impressions) threshold, and 16 of those 17 are also currently declining. This confirms the 0.1% figures weren't a bug — the impressions_90d >= 500 bar is simply strict for this sample size, leaving a small but almost entirely "already declining" pool. This threshold combination won't hold up as a standalone signal at this scale; it's a candidate to loosen (e.g. lower the impressions bar, or widen the staleness window) once I move to the ~79M-row warehouse release, where volume will support tighter thresholds without starving the pool.

**Data note:** this lane will later need care with rate columns (ctr, engagement_rate, etc. are
×100 percentages, not 0-1 fractions), avg_position == 0 meaning "no data" rather than rank
zero, and content_id/client_id being pseudonyms usable only for grouping/splitting, never as
model features. None of these affect the three numbers above, but they'll matter once I
start building features in later weeks.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

##  **Careful words: what I can and can't claim**

**What I can say:** this work is observed, measured, and directional. I can say a page shows measurable signs of decline or opportunity based on real signals — impressions, position, freshness, engagement — and that a ranked review list built this way outperforms a fixed hand-written rule on this dataset (as shown by the Precision@50 comparison in Notebook 1: 0.240 vs 0.740). Recommendations are decision-support: they help a human reviewer prioritize their limited time, not replace their judgment.

**What I can never say:** I cannot claim I've reverse-engineered or predicted Google's ranking algorithm. I cannot claim that refreshing a flagged page *causes* recovery — proving causation would require a controlled experiment (e.g. before/after with a held-out control group), which this dataset does not provide. A "declining" label describes a current pattern, not a guaranteed future outcome. Any client-identifying detail (names, domains, URLs, raw queries) must never appear in outputs — only pseudonymized IDs and aggregated metrics.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.